# 00 Data inspection
Goal: understand the data before cleaning it.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 150)

DATA_DIR = Path.cwd().parent / "data" / "raw"
print("Looking for data in:", DATA_DIR)
print("Files found:", sorted(p.name for p in DATA_DIR.glob("*.csv")))

In [ ]:
customers = pd.read_csv(DATA_DIR / "olist_customers_dataset.csv")
orders    = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items     = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments  = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews   = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")
products  = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
category_translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

tables = {
    "customers": customers, "orders": orders, "items": items,
    "payments": payments, "reviews": reviews, "products": products,
    "category_translation": category_translation,
}

In [ ]:
for name, df in tables.items():
    print("=" * 70)
    print(f"{name.upper()}  |  rows: {df.shape[0]:,}  |  columns: {df.shape[1]}")
    print("-" * 70)
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    display(summary)

In [ ]:
display(orders.head())
display(customers.head())

In [ ]:
print("Rows in customers table:        ", len(customers))
print("Unique customer_id:             ", customers["customer_id"].nunique())
print("Unique customer_unique_id:      ", customers["customer_unique_id"].nunique())

orders_per_customer = (
    orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
          .groupby("customer_unique_id")["order_id"].nunique()
)
print("\nDistribution of orders per unique customer:")
print(orders_per_customer.value_counts().sort_index().head(10))
print(f"\nShare of customers with exactly 1 order: {(orders_per_customer == 1).mean():.1%}")

In [ ]:
print(orders["order_status"].value_counts())

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
print("\nFirst order:", orders["order_purchase_timestamp"].min())
print("Last order: ", orders["order_purchase_timestamp"].max())